# Tiền xử lý dữ liệu VnExpress — HARNN

**Input:** `raw_data.json` — dữ liệu crawl thô  
**Output:** `dataset.json` · `vocab.json` · `label_map.json`

| Cell | Nhiệm vụ |
|------|----------|
| 1 | Cấu hình đường dẫn & tham số |
| 2 | Nạp logic nhãn từ `label_merge_optimized.py` |
| 3 | Kiểm tra nhanh logic nhãn mới |
| 4 | Hàm text processing |
| 5 | Load → tokenize → encode label |
| 6 | Build vocab → lưu file |
| 7 | Kiểm tra kết quả |


## Cell 1 — Cấu hình


In [1]:
import json, re
from collections import Counter, defaultdict
from pathlib import Path

# ── Đường dẫn ─────────────────────────────────────────────────────────────
RAW_FILE = r'C:\Users\Admin\Documents\nlp\NLP_Project\data\raw\raw_data1.json' 

PROJECT_DIR = Path(r'C:\Users\Admin\Documents\nlp\NLP_Project')
PROCESS_DIR = PROJECT_DIR / 'data' / 'process_data'
OUT_DATASET = PROCESS_DIR / 'dataset.json'
OUT_VOCAB = PROCESS_DIR / 'vocab.json'
OUT_LABEL_MAP = PROCESS_DIR / 'label_map.json'
STOPWORDS_FILE = PROJECT_DIR / 'data' / 'dictionary' / 'vietnamese-stopwords.txt'

# ── Tham số ───────────────────────────────────────────────────────────────
MIN_TOKENS      = 20   # bỏ bài có ít hơn N token
VOCAB_MIN_COUNT = 3    # bỏ từ xuất hiện ít hơn N lần

# Tạo thư mục output tự động: data/process_data
PROCESS_DIR.mkdir(parents=True, exist_ok=True)

print('Config OK')
print(f'Input (absolute): {RAW_FILE}')
print(f'Output folder   : {PROCESS_DIR}')


Config OK
Input (absolute): C:\Users\Admin\Documents\nlp\NLP_Project\data\raw\raw_data1.json
Output folder   : C:\Users\Admin\Documents\nlp\NLP_Project\data\process_data


## Cell 2 — Nạp logic nhãn từ file ngoài
Sử dụng toàn bộ `HIERARCHY`, `resolve_labels`, `LABEL_MAP` từ `label_merge_optimized.py`.


In [2]:
import importlib.util

LABEL_LOGIC_FILE = PROJECT_DIR / 'notebooks' / 'label_merge_optimized.py'
spec = importlib.util.spec_from_file_location('label_merge_optimized', LABEL_LOGIC_FILE)
label_logic = importlib.util.module_from_spec(spec)
spec.loader.exec_module(label_logic)

# Export các biến/hàm cần dùng cho pipeline
HIERARCHY = label_logic.HIERARCHY
L1_SET = label_logic.L1_SET
L2_SET = label_logic.L2_SET
L3_SET = label_logic.L3_SET
L2_TO_L1 = label_logic.L2_TO_L1
L3_TO_L2 = label_logic.L3_TO_L2
resolve_labels = label_logic.resolve_labels
LABEL_MAP = label_logic.LABEL_MAP

print(f'Loaded label logic from: {LABEL_LOGIC_FILE}')
print(f"L1: {len(LABEL_MAP['l1'])}  L2: {len(LABEL_MAP['l2'])}  L3: {len(LABEL_MAP['l3'])}")


Loaded label logic from: C:\Users\Admin\Documents\nlp\NLP_Project\notebooks\label_merge_optimized.py
L1: 13  L2: 47  L3: 33


## Cell 3 — Kiểm tra nhanh logic nhãn mới
Chạy test nhanh để xác nhận import thành công và hàm hoạt động đúng.


In [3]:
# Test nhanh logic nhãn mới
cases = [
    ('Thời sự',             'Tin tức',        ''),
    ('Sức khỏe',            'Tin tức',        ''),
    ('Thể thao',            'Ngoại hạng Anh', ''),
    ('Thể thao',            'Bóng đá',        'La Liga'),
    ('Xe',                  'Cầm lái',        'Tình huống'),
    ('Kinh doanh',          'Ebank',          ''),
    ('Khoa học công nghệ',  'Tin tức',        ''),
]

print(f'{"L1":<20} {"L2":<20} {"L3":<20} -> {"Out L1":<20} {"Out L2":<20} {"Out L3"}')
print('-' * 110)
for l1, l2, l3 in cases:
    out_l1, out_l2, out_l3 = resolve_labels(l1, l2, l3)
    print(f'{l1:<20} {l2:<20} {l3:<20} -> {out_l1:<20} {out_l2:<20} {out_l3}')


L1                   L2                   L3                   -> Out L1               Out L2               Out L3
--------------------------------------------------------------------------------------------------------------
Thời sự              Tin tức                                   -> Thời sự              Dân sinh             
Sức khỏe             Tin tức                                   -> Sức khỏe             Tin tức sức khỏe     
Thể thao             Ngoại hạng Anh                            -> Thể thao             Bóng đá              Các giải khác
Thể thao             Bóng đá              La Liga              -> Thể thao             Bóng đá              Các giải khác
Xe                   Cầm lái              Tình huống           -> Xe                   Cầm lái              Kỹ năng lái
Kinh doanh           Ebank                                     -> Kinh doanh           Thị trường           
Khoa học công nghệ   Tin tức                                   -> Khoa học công ngh

## Cell 4 — Text processing


In [4]:
STOPWORDS_DASH_FILE = PROJECT_DIR / 'data' / 'dictionary' / 'vietnamese-stopwords-dash.txt'

with open(STOPWORDS_FILE,      encoding='utf-8') as f:
    sw1 = {line.strip() for line in f if line.strip()}
with open(STOPWORDS_DASH_FILE, encoding='utf-8') as f:
    sw2 = {line.strip() for line in f if line.strip()}

STOPWORDS = sw1 | sw2
print(f'File gốc  : {len(sw1)} từ')
print(f'File dash : {len(sw2)} từ')
print(f'Tổng      : {len(STOPWORDS)} từ')


def clean(text: str) -> str:
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()


def tokenize(text: str) -> list[str]:
    try:
        from underthesea import word_tokenize
        tokens = word_tokenize(clean(text), format='text').split()
    except ImportError:
        tokens = clean(text).split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


def multihot(label: str, idx_map: dict) -> list[int]:
    vec = [0] * len(idx_map)
    if label and label in idx_map:
        vec[idx_map[label]] = 1
    return vec


sample = 'Cứu được người đuối nước khi livestream lúc 12h30 ngày 17/3'
print(f'\nInput : {sample}')
print(f'Tokens: {tokenize(sample)}')


File gốc  : 1942 từ
File dash : 1942 từ
Tổng      : 3513 từ

Input : Cứu được người đuối nước khi livestream lúc 12h30 ngày 17/3


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokens: ['cứu', 'đuối', 'livestream']


## Cell 5 — Load → tokenize → encode label


In [5]:
with open(RAW_FILE, encoding='utf-8') as f:
    raw = json.load(f)
print(f'Loaded {len(raw):,} bài')

articles = []
skipped  = 0

for i, a in enumerate(raw):
    # Resolve nhãn
    l1, l2, l3 = resolve_labels(
        a.get('label_l1', ''),
        a.get('label_l2', ''),
        a.get('label_l3', ''),
    )
    if not l1:
        skipped += 1
        continue

    # Tokenize
    tokens = tokenize(a.get('title', '') + ' ' + a.get('content', ''))
    if len(tokens) < MIN_TOKENS:
        skipped += 1
        continue

    articles.append({
        'url':           a.get('url', ''),
        'title':         a.get('title', ''),
        'tokens':        tokens,
        'labels_l1':     [l1],
        'labels_l2':     [l2] if l2 else [],
        'labels_l3':     [l3] if l3 else [],
        'vec_l1':        multihot(l1, LABEL_MAP['l1']),
        'vec_l2':        multihot(l2, LABEL_MAP['l2']),
        'vec_l3':        multihot(l3, LABEL_MAP['l3']),
        'is_multilabel': False,
    })

    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{len(raw)}')

print(f'\nHợp lệ : {len(articles):,}')
print(f'Bỏ qua : {skipped:,}')


Loaded 11,799 bài
  1000/11799
  2000/11799
  3000/11799
  4000/11799
  5000/11799
  6000/11799
  7000/11799
  8000/11799
  9000/11799
  10000/11799
  11000/11799

Hợp lệ : 11,787
Bỏ qua : 12


## Cell 6 — Build vocab → lưu file


In [6]:
# Build vocab
counter = Counter(t for a in articles for t in a['tokens'])
vocab   = {'<PAD>': 0, '<UNK>': 1}
for word, cnt in counter.most_common():
    if cnt >= VOCAB_MIN_COUNT:
        vocab[word] = len(vocab)
print(f'Vocab: {len(vocab):,} từ (min_count={VOCAB_MIN_COUNT})')

# Lưu
with open(OUT_DATASET,   'w', encoding='utf-8') as f: json.dump(articles,  f, ensure_ascii=False, indent=2)
with open(OUT_VOCAB,     'w', encoding='utf-8') as f: json.dump(vocab,     f, ensure_ascii=False, indent=2)
with open(OUT_LABEL_MAP, 'w', encoding='utf-8') as f: json.dump(LABEL_MAP, f, ensure_ascii=False, indent=2)

print(f'\n✓ {OUT_DATASET}')
print(f'✓ {OUT_VOCAB}')
print(f'✓ {OUT_LABEL_MAP}')


Vocab: 47,164 từ (min_count=3)

✓ C:\Users\Admin\Documents\nlp\NLP_Project\data\process_data\dataset.json
✓ C:\Users\Admin\Documents\nlp\NLP_Project\data\process_data\vocab.json
✓ C:\Users\Admin\Documents\nlp\NLP_Project\data\process_data\label_map.json


## Cell 7 — Kiểm tra kết quả


In [7]:
total   = len(articles)
has_l2  = sum(1 for a in articles if a['labels_l2'])
has_l3  = sum(1 for a in articles if a['labels_l3'])

print(f'Tổng bài   : {total:,}')
print(f'Có L2      : {has_l2:,} ({has_l2/total*100:.1f}%)')
print(f'Có L3      : {has_l3:,} ({has_l3/total*100:.1f}%)')
print(f'Vocab size : {len(vocab):,}')
print(f'vec_l1 dim : {len(articles[0]["vec_l1"])}')
print(f'vec_l2 dim : {len(articles[0]["vec_l2"])}')
print(f'vec_l3 dim : {len(articles[0]["vec_l3"])}')

# Phân phối L1
l1_dist = Counter(a['labels_l1'][0] for a in articles)
print(f'\nPhân phối L1:')
mx = max(l1_dist.values())
for lb, cnt in sorted(l1_dist.items(), key=lambda x: -x[1]):
    bar = '█' * (cnt * 30 // mx)
    print(f'  {lb:<22} {cnt:>5}  {bar}')

# 3 bài mẫu
print('\n3 bài mẫu:')
for a in articles[:3]:
    print(f"  {a['title'][:55]}")
    print(f"  L1={a['labels_l1']}  L2={a['labels_l2']}  L3={a['labels_l3']}")
    print(f"  tokens={a['tokens'][:6]}  vec_l1 sum={sum(a['vec_l1'])}")
    print()


Tổng bài   : 11,787
Có L2      : 9,585 (81.3%)
Có L3      : 2,603 (22.1%)
Vocab size : 47,164
vec_l1 dim : 13
vec_l2 dim : 47
vec_l3 dim : 33

Phân phối L1:
  Giải trí                1356  ██████████████████████████████
  Sức khỏe                1143  █████████████████████████
  Thể thao                1006  ██████████████████████
  Thời sự                 1002  ██████████████████████
  Khoa học công nghệ       997  ██████████████████████
  Giáo dục                 881  ███████████████████
  Thế giới                 867  ███████████████████
  Pháp luật                818  ██████████████████
  Du lịch                  818  ██████████████████
  Khoa học                 804  █████████████████
  Đời sống                 750  ████████████████
  Kinh doanh               689  ███████████████
  Xe                       656  ██████████████

3 bài mẫu:
  Cứu được người đuối nước khi livestream
  L1=['Thời sự']  L2=[]  L3=[]
  tokens=['cứu', 'đuối', 'livestream', 'hà_tuổi', 'trú', 'thôn']  vec_l1